In [24]:
import pyvisa
#rm = pyvisa.ResourceManager()
rm = pyvisa.ResourceManager('@py')

instrumentos = rm.list_resources()
print("Instrumentos encontrados:", instrumentos)

try:
    instrumento = rm.open_resource('ASRL4::INSTR')
    ID = instrumento.query('*IDN?')
    print("ID del instrumento:", ID)
    # Cerrar la conexión para liberar recursos.
    instrumento.close()
except Exception as e:
    print("Error al conectar con el instrumento:", e)

print("\n=== ADQUISICIÓN DE DATOS DE MEMORIA ===")
try:
    instrumento = rm.open_resource('ASRL4::INSTR')
    
    memoria = instrumento.query(':acquire1:memory?')
    #memoria = instrumento.query_binary_values(':acquire1:memory?', datatype='f', is_big_endian=True)
    #memoria = instrumento.query_ascii_values(':acquire1:memory?')
    #memoria = instrumento.read_raw(':acquire1:memory?')
    
    instrumento.close()
    
except Exception as e:
    print("Error al consultar configuración:", e)

rm.close()
print("\n=== OPERACIÓN COMPLETADA ===")
print("Conexiones cerradas correctamente")

Instrumentos encontrados: ('ASRL4::INSTR',)
ID del instrumento: GW,GDS-1102A-U,GES170725,V1.14


=== ADQUISICIÓN DE DATOS DE MEMORIA ===
Error al consultar configuración: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.

=== OPERACIÓN COMPLETADA ===
Conexiones cerradas correctamente


In [ ]:
import pyvisa
#rm = pyvisa.ResourceManager()
rm = pyvisa.ResourceManager('@py')

instrumentos = rm.list_resources()
print("Instrumentos encontrados:", instrumentos)
data=[]
try:
    instrumento = rm.open_resource('ASRL4::INSTR')
    ID = instrumento.query('*IDN?')
    print("ID del instrumento:", ID)
    # Cerrar la conexión para liberar recursos.
    instrumento.close()
except Exception as e:
    print("Error al conectar con el instrumento:", e)

print("\n=== ADQUISICIÓN DE DATOS DE MEMORIA ===")
try:
    instrumento = rm.open_resource('ASRL4::INSTR')
    
    instrumento.write(":ACQUIRE1:MEMORY?")
    #header = instrumento.read_bytes(6).decode('ascii')  # deber ser "#48008"
    #length = instrumento(header[2:])                    # aquí 8008
    #instrumento.write(":ACQUIRE1:MEMORY?")
    data = instrumento.read_bytes(8008)
    
    #header2 = data[:7].decode('ascii')
    #print("Header recibido:", header)
    #print("Header2 recibido:", header2)
    print("Data:", data)
    instrumento.close()
except Exception as e:
    print("Error al consultar configuración:", e)

rm.close()
print("\n=== OPERACIÓN COMPLETADA ===")
print("Conexiones cerradas correctamente")

Instrumentos encontrados: ('ASRL4::INSTR',)
ID del instrumento: GW,GDS-1102A-U,GES170725,V1.14


=== ADQUISICIÓN DE DATOS DE MEMORIA ===
Error al consultar configuración: MessageBasedResource.read_bytes() missing 1 required positional argument: 'count'

=== OPERACIÓN COMPLETADA ===
Conexiones cerradas correctamente


In [ ]:
import pyvisa
from pyvisa.constants import StopBits, Parity, ControlFlow

rm = pyvisa.ResourceManager('@py')

print("Instrumentos encontrados:", rm.list_resources())

# ---------- Parámetros comunes ----------
READ_TIMEOUT_MS = 5000           # 30 s de timeout (ajustá según tu equipo)
SERIAL_BAUD     = 115200         # ajustá al baud real del instrumento
CHUNK_BYTES     = 1024 * 9    # 1 MB para transferencias grandes

try:
    inst = rm.open_resource('ASRL4::INSTR')

    # ---- Configuración de la sesión VISA (ASRL = serial) ----
    inst.timeout = READ_TIMEOUT_MS             # timeout para lecturas/escrituras (ms)
    inst.baud_rate = SERIAL_BAUD               # baudrate real del equipo
    inst.data_bits = 8
    inst.stop_bits = StopBits.one
    inst.parity    = Parity.none
    inst.flow_control = ControlFlow.xon_xoff

    # Para SCPI de texto:
    inst.write_termination = '\n'
    inst.read_termination  = '\n'

    # Tamaño de bloque para lecturas largas
    inst.chunk_size = CHUNK_BYTES

    # --------- Identificación ---------
    ID = inst.query('*IDN?')
    print("ID del instrumento:", ID)

    # --------- Lectura de memoria (ASCII) ---------
    # Si el comando devuelve texto (CSV/valores):
    #try:
        # Un pequeño delay post-write a veces ayuda con equipos lentos
    #    memoria_ascii = inst.query(':acquire1:memory?', delay=0.2)
    #    print("Leí memoria ASCII (longitud):", len(memoria_ascii))
    #except Exception as e:
    #    print("Fallo lectura ASCII:", e)

    # --------- Lectura de memoria (BINARIO) ---------
    # Para binario, desactivar el terminador de lectura (muy importante)
    try:
        inst.read_termination = None
        # En binario conviene separar write y read para no depender de query()
        inst.write(':acquire1:memory?')
        # read_raw lee hasta que el instrumento cierre/termine la transferencia
        datos_bin = inst.read_bytes(8014) # Maximo numero de byte que funciona correctamente.
        print("Leí memoria BIN (bytes):", len(datos_bin))
        print("Datos BIN (primeros 64 bytes):", datos_bin[:64])
    except Exception as e:
        print("Fallo lectura BINARIA:", e)
    finally:
        # Restaurar terminador para futuras lecturas de texto
        inst.read_termination = '\n'

    inst.close()

except Exception as e:
    print("Error al conectar/operar con el instrumento:", e)
finally:
    rm.close()
    print("\n=== OPERACIÓN COMPLETADA ===")

Instrumentos encontrados: ('ASRL4::INSTR',)
ID del instrumento: GW,GDS-1102A-U,GES170725,V1.14
Leí memoria BIN (bytes): 8014
Datos BIN (primeros 64 bytes): b'#480085\x867\xbd\x01\x10\x00\x00\x003\x003\x003\x002\x002\x002\x002\x002\x002\x002\x002\x003\x003\x002\x002\x002\x002\x002\x002\x004\x004\x002\x002\x001\x001'

=== OPERACIÓN COMPLETADA ===
